# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# DONE: Import the necessary libs
# For example: 
import os
import json
import chromadb
from tavily import TavilyClient
from pydantic import BaseModel

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"

In [4]:
# DONE: Load environment variables
load_dotenv(find_dotenv(), override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
# DONE: Create retrieve_game tool
# It should use chroma client and collection you created
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

@tool
def retrieve_game(query: str):
    """Semantic search: Finds most results in the vector DB

    args:
    - query: a question about game industry.
    """
    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=['documents', 'metadatas', 'distances']
    )
    output = []
    for doc, metadata, distance in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        output.append({
            'Platform': metadata.get('Platform'),
            'Name': metadata.get('Name'),
            'YearOfRelease': metadata.get('YearOfRelease'),
            'Description': metadata.get('Description'),
            'Distance': distance,
            'Document': doc,
        })
    return output

#### Evaluate Retrieval Tool

In [6]:
# DONE: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

class EvaluationReport(BaseModel):
    useful: bool
    description: str

@tool
def evaluate_retrieval(question: str, retrieved_docs: list[dict]):
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    """
    llm = LLM(model="gpt-4o-mini")
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n"
        f"Retrieved docs: {retrieved_docs}"
    )
    response = llm.invoke(input=prompt, response_format=EvaluationReport)
    parser = PydanticOutputParser(model_class=EvaluationReport)
    return parser.parse(response).model_dump()

#### Game Web Search Tool

In [7]:
# DONE: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(question: str):
    """Search the web for game industry questions.

    args:
    - question: a question about game industry.
    """
    return tavily_client.search(query=question, max_results=3)

### Agent

In [8]:
# DONE: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

tools = [retrieve_game, evaluate_retrieval, game_web_search]
instructions = "You are UdaPlay, an AI research agent for the video game industry. Use the available tools when needed."
agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=tools,
    temperature=0.0,
)

In [9]:
# DONE: Invoke your agent
run = agent.invoke("When Pokémon Gold and Silver was released?")
print(run.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Pokémon Gold and Silver were released in Japan on November 21, 1999. They were later released in North America on October 15, 2000, and in Europe on April 6, 2001.


In [10]:
run2 = agent.invoke("Which one was the first 3D platformer Mario game?")
print(run2.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
The first 3D platformer Mario game is **Super Mario 64**, which was released in 1996 for the Nintendo 64. It is notable for being the first Super Mario game to feature 3D graphics and for its innovative gameplay that included a fully integrated camera system.


In [11]:
run3 = agent.invoke("Was Mortal Kombat X released for PlayStation 5?")
print(run3.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Mortal Kombat X was not specifically released for the PlayStation 5. However, it is playable on the PS5 through backward compatibility, meaning you can play the PlayStation 4 version of the game on the PS5. Some features available on the PS4 version may be absent when played on the PS5.


In [12]:
run4 = agent.invoke("Who publishes Halo Infinite?")
print(run4.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Halo Infinite is published by **Xbox Game Studios**. The game was developed by 343 Industries and was released on December 8, 2021.


In [13]:
run5 = agent.invoke("What games are available on PlayStation 5?")
print(run5.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
There are many games available on PlayStation 5, including:

1. **Demon's Souls**
2. **Spider-Man: Miles Morales**
3. **Ratchet & Clank: Rift Apart**
4. **Returnal**
5. **Horizon Forbidden West**
6. **Final Fantasy VII Remake Intergrade**
7. **Resident Evil Village**
8. **God of War Ragnarök**
9. **Gran Turismo 7**
10. **Elden Ring**

Additionally, many PS4 games are playable on PS5 through backward compatibility. For a comprehensive list of all available games, you can check the [Wikipedia page on PlayStation 5 games](https://en.wikipedia.org/wiki/List_of_PlayStation_5_games) or the [PlayStation Store](https://www.playstation.com/en-ca/ps5/games).


In [14]:
run6 = agent.invoke("Tell me about Gran Turismo 5")
print(run6.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
**Gran Turismo 5** is a sim racing video game developed by Polyphony Digital and published by Sony Computer Entertainment for the PlayStation 3. It was released on November 24, 2010, in Europe and North America, and on November 25, 2010, in Japan and Australasia. This game is the fifth main installment in the Gran Turismo series and the tenth overall.

### Key Features:
- **Damage Modeling**: Gran Turismo 5 introduced both mechanical and external damage modeling, including a real-time deformation engine that processes model deformation based on speed and angle of impact.
- **Vehicle Types**: The game features "premium" and "standard" vehicles. Premium vehicles are more detailed and include a fully detailed cockpit view, while standa

In [15]:
run7 = agent.invoke("What is the latest FIFA game?")
print(run7.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
The latest FIFA game is **FIFA 23**, which was released as the last installment under the partnership between EA Sports and FIFA. It features various modes, including World Cup and Women's World Cup modes, and for the first time in the franchise, it includes playable women's domestic leagues.

However, due to a failure to reach an agreement over licensing fees, FIFA 23 is the final game to be released under the FIFA name. EA Sports has since transitioned to releasing football games under the title **EA Sports FC**, with the first game in this new series expected to be released in 2023.


In [16]:
run8 = agent.invoke("What is Wii Sports about?")
print(run8.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
**Wii Sports** is a sports simulation video game developed and published by Nintendo for the Wii console, released in 2006. It serves as a collection of five sports simulations designed to showcase the motion-sensing capabilities of the Wii Remote. The sports included are:

1. **Tennis**
2. **Baseball**
3. **Bowling**
4. **Golf**
5. **Boxing**

### Key Features:
- **Motion Controls**: Players use the Wii Remote to mimic real-life actions, such as swinging a racket in tennis or rolling a bowling ball, making the gameplay intuitive and accessible.
- **Simplified Rules**: The rules for each sport are simplified to make them easy for new players to understand and enjoy.
- **Mii Characters**: Players can use their own Mii avatars, which 

In [17]:
run9 = agent.invoke("What is the best-selling Nintendo Switch game?")
print(run9.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
The best-selling Nintendo Switch game is **Mario Kart 8 Deluxe**, which has sold approximately 71.53 million copies worldwide. It is known for its engaging gameplay, accessibility, and appeal to both casual and competitive players. Other top-selling titles on the platform include **Animal Crossing: New Horizons** and **Super Smash Bros. Ultimate**.


In [18]:
run10 = agent.invoke("What is the latest Call of Duty game?")
print(run10.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
The latest Call of Duty game is **Call of Duty: Black Ops 7**, which was released on November 14, 2025. Following that, the next installment, **Call of Duty: Modern Warfare 4**, is scheduled for release on October 23, 2026.


### (Optional) Advanced

In [19]:
# DONE: Update your agent with long-term memory
from lib.memory import LongTermMemory
from lib.vector_db import VectorStoreManager

long_term_memory = LongTermMemory(VectorStoreManager(OPENAI_API_KEY))
agent.long_term_memory = long_term_memory
# DONE: Convert the agent to be a state machine, with the tools being pre-defined nodes
state_machine = agent.workflow
print(type(state_machine).__name__)

StateMachine
